# 22 septembre — Types abstraits et structures de données

Cette fiche rassemble les notes de séance et des **compléments de révision** pour rendre les raisonnements et les programmes compréhensibles. Les corrections proposées expliquent les essais présents dans les notes ; elles ne constituent pas une transcription supplémentaire du cours.

[TD 2 — Récursivité et structures immuables](TD2.pdf) · [TD 3 — Ensembles](TD3.pdf) · [Accueil du dossier](note.md)

## Parcours

1. Types abstraits et représentations concrètes.
2. Mutabilité, références et tuples.
3. Arguments par défaut.
4. Récursivité et méthode d'analyse de complexité.
5. TD 2, exercice 1 : somme récursive et limite de récursion.
6. TD 2 : repères sur les exercices suivants, puis exercice 4 expliqué et corrigé.
7. TD 3, exercice 1 : ensembles d'entiers représentés par des booléens.

**Utilisation :** exécuter les cellules de code dans l'ordre, à partir d'un noyau neuf. Les essais erronés utiles sont conservés dans les explications ; les cellules exécutables contiennent les versions corrigées. L'expérience qui relève fortement la limite de récursion est présentée séparément, dans le texte.

## 1. Type abstrait et structure de données

Un **type abstrait de données (TAD)** décrit des valeurs, des opérations et leurs propriétés, indépendamment de leur implantation. Une **structure de données** est une représentation concrète permettant de réaliser ces opérations.

Pour le TAD liste :

| Opération | Rôle |
| --- | --- |
| `lst_vide()` | Construire une liste vide. |
| `est_vide(lst)` | Tester si la liste est vide. |
| `ajouter(elm, lst)` | Construire une liste ayant `elm` en tête et `lst` pour reste. |
| `tete(lst)` | Lire le premier élément d'une liste non vide. |
| `reste(lst)` | Lire la liste privée de sa tête, pour une liste non vide. |

Par exemple, les opérations doivent vérifier `tete(ajouter(x, lst)) == x` et `reste(ajouter(x, lst)) == lst`. Ces propriétés décrivent ce que l'utilisateur de la liste peut attendre.

Une liste abstraite n'est pas nécessairement un objet Python de type `list`. Dans l'exercice 4, on utilise des **tuples imbriqués** : `()` pour la liste vide et `(tete, reste)` sinon. La suite des valeurs 1, 2, 3 se représente par `(1, (2, (3, ())))`.

L'interface décrit ce que font les opérations ; la représentation détermine comment elles le font et leur coût. Ici, `ajouter` construit une nouvelle cellule et réutilise le reste existant, sans le recopier.

## 2. Mutabilité et références

Un objet **mutable** peut être modifié après sa création. Un objet **immuable** ne peut pas être modifié après sa création.

| Types Python | Propriété |
| --- | --- |
| `int`, `float`, `bool`, `str`, `tuple` | Immuables |
| `list`, `dict`, `set` | Mutables |

Une variable désigne un objet. L'affectation `b = a` ne copie pas l'objet : les deux noms désignent le même objet. **Réaffecter un nom** et **modifier un objet** sont deux opérations différentes.

In [ ]:
a = [1, 2]
b = a
b.append(3)
print(a)  # [1, 2, 3] : modification de l'objet partagé
b = [9]
print(a)  # [1, 2, 3] : seule la référence portée par b a changé

### Un tuple peut contenir un objet mutable

Un tuple fixe les références qu'il contient : on ne peut remplacer aucun de ses éléments, **même par une valeur du même type**. En revanche, un objet mutable référencé par le tuple peut être modifié. L'immuabilité du tuple n'implique donc pas celle de tout son contenu.

Par ailleurs, `==` compare les valeurs, tandis que `is` teste si deux expressions désignent **le même objet**. Pour comparer des nombres ou le contenu de deux listes, utiliser normalement `==`.

Ces distinctions sont précisées dans le [modèle de données de Python](https://docs.python.org/3.13/reference/datamodel.html#objects-values-and-types).

In [ ]:
t = ([1, 2], "cours")
t[0].append(3)
print(t)  # ([1, 2, 3], 'cours')
# t[0] = [9]  # TypeError : remplacement d'un élément du tuple interdit

## 3. Arguments par défaut : attention aux objets mutables

Lors de l'exécution d'une instruction `def`, Python évalue les **expressions des valeurs par défaut**. Le corps de la fonction n'est exécuté qu'à l'appel.

Dans l'exemple des notes, `print(1)` s'exécute à la définition de `f`, même si `f` n'est jamais appelée. Comme `print` renvoie `None`, la valeur par défaut de `a` est la liste `[None]`.

Cette liste est conservée avec la fonction et réutilisée à chaque appel omettant l'argument. Ce n'est pas une nouvelle instance à chaque appel, ni une variable « statique » au sens strict.

In [ ]:
def f(a=[print(1)]):
    return 0

# La définition affiche 1.
# f() renverrait 0 sans afficher de nouveau 1.

Si une fonction modifie une liste utilisée comme valeur par défaut, les modifications persistent entre les appels. C'est le piège de l'exercice 2 avec `inv=[]`.

Pour créer une nouvelle liste à chaque appel qui omet l'argument, utiliser `None`, puis créer la liste dans le corps de la fonction. Un argument explicitement fourni reste utilisé tel quel : si l'appelant transmet sa propre liste, la fonction ci-dessous la modifie.

La [documentation Python sur les valeurs par défaut](https://docs.python.org/3/tutorial/controlflow.html#default-argument-values) explique ce mécanisme.

In [ ]:
def exemple(valeur, elements=None):
    if elements is None:
        elements = []
    elements.append(valeur)
    return elements

print(exemple(1))  # [1]
print(exemple(2))  # [2]

## 4. Récursivité et complexité

Une fonction récursive s'appelle elle-même. Pour justifier sa terminaison, repérer un **cas de base** et un **variant** : une quantité entière positive ou nulle qui diminue strictement à chaque appel récursif.

Pour la somme d'une liste :

- le cas de base est la liste vide, dont la somme vaut 0 ;
- un appel traite un élément puis délègue la somme des suivants ;
- le nombre d'éléments restant à traiter diminue.

Exemple : `somme([2, 5]) = 2 + somme([5]) = 2 + (5 + somme([])) = 7`. Les appels s'empilent en descendant jusqu'au cas de base, puis les résultats reviennent dans l'ordre inverse.

### Coûts à distinguer

Dans le modèle usuel des listes de CPython, `len(t)` et `t[i]` sont en temps constant. Une tranche `t[1:]` crée une nouvelle liste : son temps et son espace sont proportionnels à sa longueur. La copie est superficielle : les éléments eux-mêmes ne sont pas recopiés.

**Méthode :** compter les appels et évaluer le travail réalisé dans chacun. On note `n` la taille de l'entrée et on suppose ici les opérations arithmétiques en temps constant. Pour des entiers arbitrairement grands, leur nombre de bits intervient aussi.

`O(n)` donne une borne supérieure asymptotique ; `Θ(n)` exprime un ordre de grandeur à la fois majoré et minoré par une constante fois `n`. Un coût linéaire double approximativement quand `n` double ; un coût quadratique est multiplié par quatre.

La pile d'appels consomme aussi de la mémoire. Une récursion peut terminer mathématiquement mais dépasser la limite de récursion de Python. En CPython, même un appel récursif terminal conserve un coût de pile : on ne bénéficie pas d'une élimination automatique de ces appels.

## 5. TD 2, exercice 1 — Somme récursive

**But :** distinguer le nombre d'appels récursifs du coût réel de chaque appel. La fonction de l'énoncé est :

In [ ]:
def somme_list(t):
    if len(t) == 0:
        return 0
    return t[0] + somme_list(t[1:])

### Question 1 — Complexité de la version avec tranches

La fonction effectue `n + 1` appels, en comptant celui sur la liste vide. Cependant, chaque appel non vide copie le reste de la liste.

Les longueurs recopiées valent `n - 1`, `n - 2`, …, `1`, `0`. Leur somme est `n(n - 1) / 2`. On peut donc écrire :

`T(n) = T(n - 1) + Θ(n)`, avec `T(0) = Θ(1)`.

Le temps total est **Θ(n²)**. Compter seulement les appels et conclure à un temps linéaire oublierait les copies.

La pile a une profondeur Θ(n). Les tranches restent également référencées par les appels en attente : leur espace cumulé atteint **Θ(n²)** dans cette version.

### Question 2 — Supprimer les copies grâce à un indice

L'indice `i` désigne le premier élément restant à additionner. La même liste `t` est transmise à tous les appels. Le variant est `len(t) - i`.

L'énoncé parle de « paramètre nommé » : on utilise ici un paramètre ayant une valeur par défaut, `i=0`, et on peut le transmettre par son nom, `i=i + 1`. Ces deux notions sont liées dans l'exemple, mais distinctes en Python.

**Contrat :** `t` est une liste de nombres et `0 <= i <= len(t)`. Un appel avec `i=2` calcule la somme à partir de la troisième case.

In [ ]:
def somme_indices(t, i=0):
    if i == len(t):
        return 0
    return t[i] + somme_indices(t, i=i + 1)

print(somme_indices([2, 5, 3]))       # 10
print(somme_indices([2, 5, 3], i=1))  # 8

### Question 3 — Cacher l'indice dans une fonction imbriquée

La fonction extérieure reçoit seulement `t`. La fonction intérieure `rec` reçoit un indice obligatoire et accède à `t`, défini dans la portée englobante. L'appel initial se fait avec l'indice 0.

Il y a Θ(n) appels et un travail constant par appel : le temps est **Θ(n)** pour cette version comme pour `somme_indices`. L'espace auxiliaire reste **Θ(n)** à cause de la pile, même si aucune tranche n'est créée.

In [ ]:
def somme_imbriquee(t):
    def rec(i):
        if i == len(t):
            return 0
        return t[i] + rec(i + 1)

    return rec(0)

### Vérifier les premières versions

Essayer d'abord des listes courtes : vide, singleton, plusieurs éléments, nombres négatifs. `sum` sert de référence pour vérifier le résultat, sans remplacer l'implantation demandée.

In [ ]:
cas = [[], [7], [1, 2, 3, 4], [-3, 1, 2]]
for valeurs in cas:
    attendu = sum(valeurs)
    for fonction in (somme_list, somme_indices, somme_imbriquee):
        assert fonction(valeurs) == attendu
    print(valeurs, "→", attendu)

### Questions 4 à 7 — Limite de récursion

La liste contenant les entiers de 0 à 10000 inclus s'écrit `list(range(10001))`. Sa somme vaut `10000 × 10001 / 2 = 50 005 000`.

Avec une limite de récursion habituelle, la version récursive lève **`RecursionError`** avant d'atteindre le cas de base. `sys.getrecursionlimit()` indique la limite effective ; elle dépend de l'environnement. Une partie de la profondeur est déjà utilisée par les appels entourant la fonction.

Pour l'expérience demandée à la question 6, voici le principe à exécuter séparément si l'environnement le permet :

```python
ancienne_limite = sys.getrecursionlimit()
try:
    sys.setrecursionlimit(max(ancienne_limite, 12000))
    print(somme_imbriquee(list(range(10001))))
finally:
    sys.setrecursionlimit(ancienne_limite)
```

Augmenter cette limite ne diminue ni le nombre d'appels ni la mémoire nécessaire. Une valeur trop élevée peut provoquer un arrêt de l'interpréteur selon la plateforme : ce n'est donc pas une solution générale. Voir la [documentation de `sys.setrecursionlimit`](https://docs.python.org/3.13/library/sys.html#sys.setrecursionlimit).

Une version itérative conserve seulement un total et un élément courant : elle calcule la somme en **Θ(n) temps et O(1) espace auxiliaire**, dans notre modèle arithmétique. Elle ne rencontre pas la limite de récursion.

In [ ]:
import sys

print("Limite de récursion actuelle :", sys.getrecursionlimit())

def somme_iterative(t):
    total = 0
    for valeur in t:
        total += valeur
    return total

# L'exécution normale du notebook ne relève pas la limite de récursion.
assert somme_iterative(range(10001)) == 50_005_000
print(somme_iterative(range(10001)))

## 6. TD 2 — Repères pour la suite

| Exercice | Idée principale |
| --- | --- |
| 2 | Inverser une liste récursivement et corriger le partage de `inv=[]`. |
| 3 | Mesurer la mémoire des objets référencés ; éviter de compter plusieurs fois le même objet grâce à `id`. |
| 4 | Implanter une liste immuable par tuples imbriqués, puis écrire ses opérations. |
| 5 | Construire une file immuable avec deux listes. |
| 6 | Rechercher, fusionner et trier avec les listes de l'exercice 4. |

Pour l'exercice 3, `sys.getsizeof(lst)` mesure le conteneur, sans additionner automatiquement les objets qu'il référence. Un ensemble d'identifiants déjà visités permet de gérer les références partagées et d'éviter de boucler sur une référence circulaire. Pour un dictionnaire, il faut examiner les clés **et** les valeurs. Voir [`sys.getsizeof`](https://docs.python.org/3.13/library/sys.html#sys.getsizeof).

### Complément pour l'exercice 2 — Pourquoi `inv=None` ?

Avec `inv=[]`, une première inversion laisse son résultat dans la liste par défaut. Un deuxième appel sans argument `inv` réutilise cette même liste et poursuit son remplissage. Avec `None`, chaque appel initial construit son propre accumulateur, puis les appels récursifs se transmettent cet accumulateur.

La version ci-dessous lit la liste de droite à gauche. Elle utilise `append`, de coût amorti constant, au lieu d'insérer chaque élément en position 0. Le temps total est Θ(n) ; la pile et la liste résultat occupent chacune Θ(n) espace.

In [ ]:
def inverse_lst(lst, i=0, inv=None):
    if inv is None:
        inv = []
    if i == len(lst):
        return inv
    inv.append(lst[len(lst) - 1 - i])
    return inverse_lst(lst, i=i + 1, inv=inv)

assert inverse_lst([1, 2, 3, 4]) == [4, 3, 2, 1]
assert inverse_lst([5, 6, 7]) == [7, 6, 5]
assert inverse_lst([]) == []

## TD 2, exercice 4 — Listes immuables

### Questions 1 et 2 — Opérations élémentaires

Une liste est soit vide, soit un couple `(tete, reste)`. Par exemple, `(7, (5, ()))` contient deux valeurs ; le tuple extérieur a toujours deux cases, quelle que soit la longueur de la liste abstraite.

**À ne pas confondre :** `lst[1:]` renvoie un tuple contenant le reste, et non le reste lui-même. Pour `(7, (5, ()))`, la tranche vaut `((5, ()),)` tandis que le reste attendu est `(5, ())`. Il faut utiliser `lst[1]` ou décomposer le couple avec `_, queue = lst`.

On choisit de lever `ValueError` pour `tete` et `reste` sur la liste vide : ces opérations n'y ont pas de résultat défini. Toutes les fonctions supposent que l'argument respecte la représentation par tuples imbriqués.

In [ ]:
def lst_vide():
    return ()

def ajouter(elm, lst):
    return (elm, lst)

def est_vide(lst):
    return lst == ()

def tete(lst):
    if est_vide(lst):
        raise ValueError("Une liste vide n'a pas de tête.")
    return lst[0]  # Équivalent : premier, _ = lst ; return premier

def reste(lst):
    if est_vide(lst):
        raise ValueError("Une liste vide n'a pas de reste.")
    return lst[1]  # Équivalent : _, queue = lst ; return queue

ancienne = ajouter(2, lst_vide())
nouvelle = ajouter(1, ancienne)
assert ancienne == (2, ())
assert nouvelle == (1, (2, ()))
assert reste(nouvelle) is ancienne  # Le reste est partagé, sans copie.

**Complexité :** `est_vide`, `ajouter`, `tete` et `reste` sont en **O(1)** sur une représentation valide. `ajouter` construit seulement un tuple de deux références ; `reste` lit une référence déjà présente. Construire la liste vide est aussi en O(1).

La structure des maillons est immuable. Si les valeurs stockées sont elles-mêmes mutables, elles peuvent néanmoins changer, comme dans l'exemple du tuple contenant une liste.

### Question 3 — Longueur

La longueur de la liste vide est 0 ; celle d'une liste non vide est `1 + longueur(reste(lst))`. Le variant est le nombre de maillons restants. Chaque maillon est visité une fois : **Θ(n) temps et Θ(n) espace de pile**. À distinguer de `len` sur une liste Python, qui est en O(1).

In [ ]:
def longueur(lst):
    if est_vide(lst):
        return 0
    return 1 + longueur(reste(lst))

### Question 4 — Convertir une liste Python

L'essai `return ajouter(lstp[0], cree_lst(lstp[1:]))` avait deux difficultés :

- il manquait le cas de base `lstp == []`, ce qui finit par provoquer `IndexError` ;
- même avec ce cas de base, les tranches successives donnent un coût quadratique.

Voici cet essai rendu correct, pour voir précisément le rôle du cas de base. Il conserve l'ordre des valeurs, mais reste en **Θ(n²)** en temps et en espace total à cause des tranches.

In [ ]:
def cree_lst(lstp):
    if len(lstp) == 0:
        return lst_vide()
    return ajouter(lstp[0], cree_lst(lstp[1:]))

assert cree_lst([1, 2, 3]) == (1, (2, (3, ())))

**Version linéaire :** construire la liste par ajouts en tête, en parcourant les valeurs de droite à gauche.

La première boucle des notes parcourait `for v in lstp`, puis ajoutait chaque valeur en tête : elle obtenait donc l'ordre inverse. Pour préserver `[7, 5, 6, 3, 2]`, on ajoute successivement `2`, `3`, `6`, `5`, puis `7`.

`reversed(lstp)` parcourt une liste Python à l'envers sans en créer une copie. La conversion coûte **Θ(n)** temps, **Θ(n)** espace pour le résultat et **O(1)** espace auxiliaire, en dehors du résultat.

In [ ]:
def lstp_vers_lst(lstp):
    lst = lst_vide()
    for valeur in reversed(lstp):
        lst = ajouter(valeur, lst)
    return lst

In [ ]:
test = lstp_vers_lst([7, 5, 6, 3, 2])
assert test == (7, (5, (6, (3, (2, ())))))
assert lstp_vers_lst([]) == lst_vide()
assert longueur(test) == 5
print(longueur(test))  # 5

### Question 5 — Somme des valeurs

On applique la même décomposition que pour la longueur, mais on ajoute la tête au lieu d'ajouter 1. La liste vide a pour somme 0. Le temps est **Θ(n)** et l'espace de pile **Θ(n)**, en supposant l'addition en temps constant.

Contrairement à `somme_list` de l'exercice 1, `reste(lst)` ne copie rien : il n'y a pas de coût caché de tranche.

In [ ]:
def somme(lst):
    if est_vide(lst):
        return 0
    return tete(lst) + somme(reste(lst))

# Écriture équivalente du corps :
# return 0 if est_vide(lst) else tete(lst) + somme(reste(lst))

In [ ]:
assert somme(lst_vide()) == 0
assert somme(test) == 23
print(somme(test))  # 23

### Question 6 — Afficher la tête à gauche

On souhaite obtenir `[7; 5; 6; 3; 2]`, et non la représentation brute en tuples. La fonction extérieure traite le cas vide ; l'auxiliaire peut donc supposer que sa liste est non vide. Il affiche le dernier élément avec le crochet fermant, et les autres avec un séparateur.

Le parcours est linéaire et la pile occupe Θ(n) espace. Le coût exact de l'affichage dépend aussi du nombre de caractères produits. `affiche` écrit dans la sortie et renvoie implicitement `None` ; elle ne renvoie pas une chaîne.

In [ ]:
def affiche(lst):
    def rec(lst):
        if est_vide(reste(lst)):
            print(tete(lst), "]", sep="")
        else:
            print(tete(lst), end="; ")
            rec(reste(lst))

    if est_vide(lst):
        print("[]")
    else:
        print("[", end="")
        rec(lst)


In [ ]:
affiche(lst_vide())             # []
affiche(ajouter(7, lst_vide()))  # [7]
affiche(test)                   # [7; 5; 6; 3; 2]

### Question 7 — Inverser avec un accumulateur

L'accumulateur `acc` contient les éléments déjà parcourus dans l'ordre inverse. À chaque appel, on retire la tête de la liste à traiter et on l'ajoute en tête de l'accumulateur.

Pour `[1, 2, 3]`, les couples « restant / accumulateur » sont : `[1, 2, 3] / []`, puis `[2, 3] / [1]`, puis `[3] / [2, 1]`, puis `[] / [3, 2, 1]`.

L'ancien appel `inverse(tete(lst), ajouter(test(lst), acc))` était incorrect : le premier argument doit être **le reste de la liste**, et l'élément ajouté doit être **sa tête**. De plus, `test` est une variable de notre exemple, pas une fonction.

Ici, la valeur par défaut `()` est immuable. Chaque ajout construit un nouveau tuple : il n'y a donc pas le partage mutable problématique de `inv=[]`.

Temps **Θ(n)**, résultat de taille **Θ(n)** et pile **Θ(n)**. Bien que l'appel soit terminal, CPython conserve les appels en attente.

In [ ]:
def inverse(lst, acc=()):
    if est_vide(lst):
        return acc
    return inverse(reste(lst), ajouter(tete(lst), acc))

assert inverse(lst_vide()) == lst_vide()
assert inverse(ajouter(7, lst_vide())) == ajouter(7, lst_vide())
assert inverse(test) == lstp_vers_lst([2, 3, 6, 5, 7])
assert inverse(inverse(test)) == test
assert test == lstp_vers_lst([7, 5, 6, 3, 2])  # L'original est conservé.
affiche(inverse(test))  # [2; 3; 6; 5; 7]

# 7. TD 3 — Ensembles

[énoncé du TD 3](TD3.pdf)

## Exercice 1 — Représentation par tableau de booléens

Un ensemble contient chaque valeur au plus une fois. L'ordre d'ajout n'a pas d'importance. Ajouter deux fois la même valeur ou supprimer une valeur déjà absente ne change pas le résultat.

**Objectif :** implanter une classe `Ensemble` pour les entiers de **0 à n inclus**, avec une liste Python de booléens dont la taille ne change pas. `Ensemble(n)` construit un ensemble vide.

### Principe de la représentation

La case d'indice `x` indique si l'entier `x` appartient à l'ensemble : `False` signifie absent et `True` signifie présent.

Il faut **n + 1 cases**, car 0 est inclus. Pour `n = 5`, l'ensemble `{1, 4}` est représenté par `[False, True, False, False, True, False]`.

La capacité est `n + 1`, le **cardinal** est le nombre de valeurs présentes, et la **somme** est l'addition de ces valeurs. Pour `{1, 4}`, le cardinal vaut 2 et la somme vaut 5.

La classe ci-dessous regroupe les réponses aux questions 1 à 4. Elle précise aussi le comportement hors du domaine : l'ajout et la suppression lèvent une exception pour une valeur invalide ; le test d'appartenance renvoie `False`.

In [ ]:
class Ensemble:
    def __init__(self, n):
        if not isinstance(n, int):
            raise TypeError("La borne doit être un entier.")
        if n < 0:
            raise ValueError("La borne doit être positive ou nulle.")
        self.__n = n
        self.__taille = 0
        self.__t = [False] * (n + 1)

    def __verifier(self, k):
        if not isinstance(k, int):
            raise TypeError("Un élément doit être un entier.")
        if not 0 <= k <= self.__n:
            raise ValueError("L'élément doit être compris entre 0 et n.")

    # Question 1 : opérations élémentaires.
    def est_vide(self):
        return self.__taille == 0

    def ajouter(self, k):
        self.__verifier(k)
        if not self.__t[k]:
            self.__t[k] = True
            self.__taille += 1

    def supprimer(self, k):
        self.__verifier(k)
        if self.__t[k]:
            self.__t[k] = False
            self.__taille -= 1

    def __contains__(self, k):
        return isinstance(k, int) and 0 <= k <= self.__n and self.__t[k]

    # Question 2 : ajout des valeurs d'une liste.
    def ajouter_lst(self, lst):
        for k in lst:
            self.ajouter(k)

    # Question 3 : somme des valeurs présentes, pas leur nombre.
    def somme(self):
        total = 0
        for k in range(self.__n + 1):
            if self.__t[k]:
                total += k
        return total

    # Question 4 : cardinal en temps constant.
    def __len__(self):
        return self.__taille

### Lire et comprendre la classe

`__init__` initialise chaque instance. `self` désigne l'objet sur lequel la méthode travaille. Les attributs `__n`, `__t` et `__taille` stockent respectivement la borne, les présences et le cardinal.

Le double soulignement initial provoque un renommage des attributs par Python ; il évite surtout les collisions de noms et ne constitue pas une protection absolue. Voir la [documentation sur les attributs internes](https://docs.python.org/3.13/tutorial/classes.html#private-variables).

`ajouter(k)` et `supprimer(k)` modifient l'objet sur place. **`__contains__` permet `k in e`**, et **`__len__` permet `len(e)`**. On écrit `k not in e` pour le test d'absence.

L'initialisation `[False] * (n + 1)` convient car les booléens sont immuables. Avec des sous-listes mutables, une multiplication peut partager la même sous-liste entre les cases.

La vérification des bornes empêche notamment un entier négatif d'accéder aux cases depuis la fin de la liste. Le test d'appartenance utilise le court-circuit de `and` : l'accès `self.__t[k]` n'a lieu que si le type et les bornes conviennent.

### Question 2 — Ajouter plusieurs valeurs

`ajouter_lst` réutilise `ajouter`. Cette réutilisation évite de dupliquer la logique et assure le bon traitement des doublons. La méthode suppose que toutes les valeurs fournies sont valides ; si une valeur invalide apparaît après plusieurs ajouts, les ajouts précédents restent effectués.

### Question 3 — Distinguer somme et cardinal

Dans l'essai `def somme(self, lst)`, le paramètre `lst` ne servait pas, `n` n'était pas défini dans la méthode et `stack += 1` comptait les présences.

La correction emploie `self.__n` et ajoute **`k`**, la valeur présente. La méthode s'appelle donc `e.somme()`, sans autre argument. Pour `{0, 2, 4}`, le cardinal vaut 3 et la somme vaut 6 : inclure 0 augmente le cardinal, mais pas la somme.

### Exemple

In [ ]:
e = Ensemble(8)
e.ajouter_lst([2, 4, 7, 8])

print(4 in e)      # True
print(3 in e)      # False
print(3 not in e)  # True
assert [k for k in range(9) if k in e] == [2, 4, 7, 8]
assert len(e) == 4
assert e.somme() == 21

e.ajouter(4)  # Un doublon ne change ni le cardinal ni la somme.
assert len(e) == 4
assert e.somme() == 21
e.supprimer(4)
assert 4 not in e
assert len(e) == 3
assert e.somme() == 17
assert -1 not in e and 9 not in e

### Complexités

On note `n` la borne supérieure du domaine et `m` le nombre de valeurs fournies à `ajouter_lst`.

| Opération | Temps |
| --- | --- |
| Construire `Ensemble(n)` | Θ(n + 1) |
| Tester l'appartenance avec `in` | O(1) |
| Ajouter ou supprimer un entier | O(1) |
| Ajouter une liste de `m` valeurs | O(m) |
| Calculer `e.somme()` | Θ(n + 1) |
| Obtenir le cardinal avec `len(e)` | O(1) |
| Tester si l'ensemble est vide | O(1) |

L'espace utilisé est **Θ(n + 1)**, même si l'ensemble contient peu d'éléments. La somme parcourt toutes les cases, y compris celles correspondant à des valeurs absentes : son coût dépend de la capacité, pas seulement du cardinal.

### Question 4 — Nombre d'éléments en temps constant

On conserve le compteur `self.__taille`, initialisé à 0.

- Un ajout l'incrémente **seulement si la valeur était absente**.
- Une suppression le décrémente **seulement si la valeur était présente**.
- `__len__` renvoie ce compteur sans parcourir le tableau.

**Invariant :** `self.__taille` reste égal au nombre de cases à `True`. Il est vrai à la création et préservé par chaque opération. `ajouter_lst` le respecte également puisqu'elle utilise `ajouter`.

Affecter `self.__len__ = k` ne convient pas : cela stockerait le dernier élément parcouru, et non le cardinal. `__len__` doit rester une méthode ; la donnée est stockée dans `__taille`.

Après une modification de la classe, réexécuter sa cellule puis recréer les objets pour disposer de la nouvelle définition.

In [ ]:
e4 = Ensemble(8)
assert e4.est_vide()
assert len(e4) == 0 and e4.somme() == 0

e4.ajouter_lst([2, 4, 4, 7, 8])
assert len(e4) == 4  # Le doublon 4 ne compte qu'une fois.
assert e4.somme() == 21

e4.supprimer(4)
assert len(e4) == 3 and e4.somme() == 17
e4.supprimer(4)  # Déjà absent : aucun changement.
e4.ajouter_lst([])
assert len(e4) == 3

zero = Ensemble(0)  # Domaine réduit à la seule valeur 0.
zero.ajouter(0)
assert 0 in zero and len(zero) == 1 and zero.somme() == 0
zero.supprimer(0)
assert zero.est_vide()
print("Exemples vérifiés : doublons, suppressions et domaine réduit à 0.")

## Pour vérifier sa compréhension

1. Pourquoi `somme_list` est-elle quadratique malgré un seul appel récursif par élément ? Parce que chaque tranche recopie le reste de la liste.
2. Pourquoi `reste` est-il constant pour les tuples imbriqués ? Parce qu'il renvoie une référence déjà stockée dans le second champ.
3. Pourquoi parcourir une liste de gauche à droite en ajoutant en tête inverse-t-il l'ordre ? Chaque nouvelle valeur passe devant toutes les précédentes.
4. Pourquoi `inv=[]` est-il problématique alors que `acc=()` convient ici ? Le premier objet est modifié et partagé entre appels ; le second reste immuable.
5. Pourquoi ajouter 0 à un ensemble peut-il changer sa longueur sans changer sa somme ? Le cardinal compte les valeurs, tandis que la somme additionne leur valeur numérique.

Les exercices non développés ici restent disponibles dans les PDF : file immuable et algorithmes sur les listes pour le TD 2 ; ensembles par bits et dictionnaires pour le TD 3.